In [1]:
import torch
import torchvision.transforms.v2 as T
from timm.data.constants import IMAGENET_INCEPTION_MEAN, IMAGENET_INCEPTION_STD
from torchvision.io import read_image 
import os,glob
import pandas as pd
from pprint import pprint as pp
import yaml
def read_yaml(fpath): 
    with open(fpath, mode="r",encoding='utf-8') as file:
        yml = yaml.load(file, Loader=yaml.Loader)
        return dict(yml)


In [2]:
class NEVA_InputLoader:
    def __init__(self, image_dir, report):
        
        self.image_dir = image_dir   
        self.report = report  

        self.MAX_BAG = 200
        self.img_size = 224
        self.transform = T.Compose([
            T.Resize(self.img_size, interpolation=T.InterpolationMode.BILINEAR, antialias=True),
            T.CenterCrop((self.img_size, self.img_size)),
            T.ConvertImageDtype(torch.float32),
            T.Normalize(mean=IMAGENET_INCEPTION_MEAN, std=IMAGENET_INCEPTION_STD),
        ])

        assert os.path.isdir(self.image_dir), f"{self.image_dir} is not a valid directory."

    @property  
    def load_image(self) -> torch.Tensor:
        patch_list = sorted(glob.glob(os.path.join(self.image_dir, "*.png")))
        num_patches = len(patch_list)

        # Validate the number of patches
        if num_patches < 2:
            raise ValueError(f"Expected at least 2 patches, but found {num_patches} in {self.image_dir}")
        if num_patches < self.MAX_BAG:
            # Warn if padding is required but within acceptable range
            print(f"Warning: Only {num_patches} patches found in {self.image_dir}. Padding to {self.MAX_BAG}.")

        
        # Load available patches
        image_tensors = []
        for patch_path in patch_list[:self.MAX_BAG]:  # Load up to MAX_BAG patches
            image_tensors.append(read_image(patch_path))
        
        # Pad with blank images if necessary
        if num_patches < self.MAX_BAG:
            blank_image = torch.zeros_like(image_tensors[0])
            for _ in range(self.MAX_BAG - num_patches):
                image_tensors.append(blank_image)


        patch_bag_tensor = torch.stack(image_tensors, dim=0)  # [200, 3, H, W]
        patch_bag_tensor = self.transform(patch_bag_tensor)  # [200, 3, 224, 224]

        assert patch_bag_tensor.shape[0] == self.MAX_BAG, f"Expected {self.MAX_BAG} patches, got {patch_bag_tensor.shape[0]}"
        assert patch_bag_tensor.shape[1] == 3, f"Expected 3 channels, got {patch_bag_tensor.shape[1]}"
        expected_size = (self.img_size, self.img_size)
        assert patch_bag_tensor.shape[2:] == expected_size, f"Expected size {expected_size}, got {patch_bag_tensor.shape[2:]}"
        
        return patch_bag_tensor  # [200, 3, self.img_size, self.img_size] 比如[200, 3, 224, 224]

    @property
    def load_report(self) -> torch.Tensor:
        """load text feature"""
        # load tokenzier for language input
        from transformers import XLMRobertaTokenizer
        from musk import utils, modeling
        # load tokenzier for language input
        tokenizer = XLMRobertaTokenizer("./musk/models/tokenizer.spm")

        report_ids, report_pads = utils.xlm_tokenizer(self.report, tokenizer, max_len=500)

        return (torch.tensor(report_ids), torch.tensor(report_pads)) 



In [10]:
test_data = NEVA_InputLoader(image_dir="../1_process_wsi/top200_patches/2020075", report="this is a demo report for testing the NEVA input loader. It should be tokenized and padded correctly.")
images = test_data.load_image
report_ids, report_pads = test_data.load_report
pp(f"Image shape: {images.shape}")
pp(f"Report IDs length: {report_ids.shape}")
pp(f"Report Pads length: {report_pads.shape}")

'Image shape: torch.Size([200, 3, 224, 224])'
'Report IDs length: torch.Size([500])'
'Report Pads length: torch.Size([500])'


In [13]:
from models import MILModel

task_type = {
    'Risk_Group': 2,  # 0: low risk, 1: high risk
    'Subtype': 3,     # 0: NB(Neuroblastoma)    1: GNB(Ganglioneuroblastoma)    2: GN(Ganglioneuroma)
    'Shimada': 2,     # 0： UnFavorable   1：Favorable
    'MKI': 3,         # 0: low risk, 1: medium risk, 2: high risk
    'NMYC': 2,        # 0: non-amplified, 1: amplified
    'CMYC': 2,        # 0: normal, 1: overexpressed
    'ALK': 2,         # 0: normal, 1: overexpressed
    '1p36': 2,        # 0: intact, 1: deleted
    '11q23': 2,       # 0: intact, 1: deleted
    'PFS': 1,         # lower than -2.901: poor prognosis, higher than -2.901 months: good prognosis
    'OS': 1,          # lower than -8.015: poor prognosis, higher than -8.015 months: good prognosis
}

TASK = 'Shimada'  # chosen from the above task_type keys, e.g., 'Risk_Group', 'Subtype', 'Shimada', 'MKI', 'NMYC', 'CMYC', 'ALK', '1p36', '11q23'

NEVA_config = read_yaml(f"./configs/NEVA.yaml")
NEVA_config['Model']['params']['num_classes'] = task_type[TASK]   

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MILModel(NEVA_config, save_path='./').cpu().to(device) 
model_wts_path = f"./neva_wts/{TASK}.pth"
assert os.path.isfile(model_wts_path), f"Model weights not found at {model_wts_path}"

raw_state = torch.load(model_wts_path, map_location=device)
state_dict = raw_state.get("state_dict", raw_state)
model.load_state_dict(state_dict, strict=False)


with torch.inference_mode():
    model.eval()
    images_ = images.unsqueeze(0).to(device)
    report_ = (report_ids.unsqueeze(0).to(device), report_pads.unsqueeze(0).to(device))  # ([1, 500], [1, 500])
    logit_, _ = model((images_, report_))  # [1, n_classes]
    print(logit_.cpu())
    print(torch.softmax(logit_.cpu(), dim=-1))


Load ckpt from hf_hub:xiangjx/musk
Position interpolate from 24x24 to 14x14
trainable params: 196,608 || all params: 674,996,225 || trainable%: 0.0291


/tmp/ipykernel_147667/2437888641.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  raw_state = torch.load(model_wts_path, map_location=device)


tensor([[-0.5973,  0.5596]])
tensor([[0.2392, 0.7608]])
